# Phase 2b: Full-Quality Training on Your Own Scene (wicker table)

Garden showed which recipe works on a free T4: **MCMC with a 1M Gaussian cap, 30k steps** (PSNR 26.99, 0.42 dB below the published 3DGS result, 35 min, 3.4 GB). Here we apply it to **your own phone video** (`scene1`), the scene that will become the digital twin in Phase 3.

**Same frames and poses as Phase 1b.** The notebook re-extracts the frames with the same settings and reuses the COLMAP model cached on Drive, so the poses and the test frames are identical. That makes the comparison with Phase 1b fair: same images, same split, only the training changes.

| | Phase 1b | This notebook |
|---|---|---|
| Strategy | default | **MCMC, 1M cap** |
| Steps | 7k | **30k** |

**First session:** COLMAP is installed (~3 min) to undistort the frames, and the training-ready scene is zipped to Drive (`datasets/scene1_ready.zip`, ~30-60 MB). **Later sessions** (Phase 3) unzip it in seconds.

⚠️ **What 30k steps can't fix:** parts of the scene the video barely saw (the background, the floor reflections in Phase 1b) stay weak. More training sharpens what was filmed; it can't invent missing views. That needs a better capture (`docs/capture_guide.md`).

⚠️ Held-out video frames sit right next to training frames, so these scores are higher than garden's and **can't be compared with garden's**. Compare them only with Phase 1b.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`, and keep the tab open.

⏱️ On a T4: install 2-5 min, scene preparation ~5 min the first time, **training ~25-40 min** (estimate: 300 small frames, 1M cap).

## 1. GPU, Drive, settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, re, subprocess, shutil
SCENE = 'scene1'            # video file name in MyDrive/3dgs-digital-twin/videos/

# Frame settings: must match Phase 1b, so the cached COLMAP model fits the frames
TARGET_FRAMES = 300
OVERSAMPLE    = 3
MAX_SIDE      = 1600
MATCHER       = 'sequential'   # only used if there's no cached COLMAP model

EXPERIMENTS = {
    'mcmc_1M':   ['mcmc', '--strategy.cap-max', '1000000'],   # the garden winner
    'mcmc_500k': ['mcmc', '--strategy.cap-max', '500000'],    # optional: half-size file for the web viewer
}
RUN_NOW = ['mcmc_1M']
STEPS   = 30_000

PROJECT = '/content/drive/MyDrive/3dgs-digital-twin'
WORK    = '/content/work'
OUT     = f'{PROJECT}/results/phase1b_{SCENE}_sfm'          # Phase 1b's folder: holds the cached COLMAP model
FIG     = f'{PROJECT}/results/phase2_{SCENE}_figures'
READY_ZIP = f'{PROJECT}/datasets/{SCENE}_ready.zip'          # training-ready scene (undistorted images + poses)
for d in [f'{PROJECT}/results', f'{PROJECT}/exports', f'{PROJECT}/datasets', f'{PROJECT}/cache', WORK, OUT, FIG]:
    os.makedirs(d, exist_ok=True)
run_name = lambda exp: f'phase2_{SCENE}_{exp}_{STEPS // 1000}k'

## 2. Install gsplat

The same two cells as the garden notebook. The second one should print `✅ gsplat CUDA loaded from the Drive cache`.

In [ ]:
import importlib.util
MARKER = '/content/.gs_deps_ok'          # /content survives a kernel restart (not a VM reset)

if not os.path.exists(MARKER):
    import torch
    np_before = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                               capture_output=True, text=True).stdout.strip()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    if 'T4' in torch.cuda.get_device_name(0):
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile', 'numpy>=2.1,<2.5'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')
    np_after = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                              capture_output=True, text=True).stdout.strip()
    if np_before != np_after:
        print(f'numpy {np_before} -> {np_after}: restarting runtime. Then "Run all" again.')
        time.sleep(2); os.kill(os.getpid(), 9)

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
# F1.5 (second fix): keep gsplat's compiled CUDA library on Drive and load it directly in later sessions
import torch, importlib.util
PREBUILT = '/content/gsplat_prebuilt'
cap = torch.cuda.get_device_capability()
KEY = f"gsplat1.5.3_torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}_sm{cap[0]}{cap[1]}"
JIT_CACHE = f'{PROJECT}/cache/gsplat_jit/{KEY}'

# 1. Patch gsplat's loader: use /content/gsplat_prebuilt/gsplat_cuda.so if it exists, otherwise behave as before
BACKEND = os.path.join(os.path.dirname(importlib.util.find_spec('gsplat').origin), 'cuda', '_backend.py')
src = open(BACKEND).read()
if 'GSPLAT_PREBUILT_DIR' not in src:
    old = '    from gsplat import csrc as _C\n'
    new = ('    _pre = os.environ.get("GSPLAT_PREBUILT_DIR", "/content/gsplat_prebuilt")\n'
           '    if os.path.exists(os.path.join(_pre, "gsplat_cuda.so")):\n'
           '        _C = _import_module_from_library("gsplat_cuda", _pre, True)\n'
           '    else:\n'
           '        from gsplat import csrc as _C\n')
    assert src.count(old) == 1, 'gsplat _backend.py changed, patch needs updating'
    open(BACKEND, 'w').write(src.replace(old, new)); print('patched gsplat loader')

# 2. Restore the cached library from Drive
os.makedirs(PREBUILT, exist_ok=True)
if not os.path.exists(f'{PREBUILT}/gsplat_cuda.so') and os.path.exists(f'{JIT_CACHE}/gsplat_cuda.so'):
    shutil.copy(f'{JIT_CACHE}/gsplat_cuda.so', PREBUILT)

def load_gsplat_cuda():
    t = time.time()
    r = subprocess.run([sys.executable, '-c', 'from gsplat.cuda._backend import _C; print("SO=", _C.__file__ if _C else None)'],
                       capture_output=True, text=True)
    so = re.search(r'SO= (.+)', r.stdout)
    return (so.group(1).strip() if so else None), time.time() - t, r

so, dt, r = load_gsplat_cuda()
if so and so.startswith(PREBUILT):
    print(f'✅ gsplat CUDA loaded from the Drive cache in {dt:.0f}s ({KEY})')
else:
    # 3. No usable cache: the import above compiled it (~15 min). Save the result for next time.
    if not so or so == 'None':
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('gsplat CUDA failed to load or compile')
    print(f'compiled gsplat CUDA in {dt/60:.1f} min -> saving to Drive')
    os.makedirs(JIT_CACHE, exist_ok=True)
    shutil.copy(so, f'{JIT_CACHE}/gsplat_cuda.so'); shutil.copy(so, f'{PREBUILT}/gsplat_cuda.so')
    so2, dt2, _ = load_gsplat_cuda()
    print(f"{'✅' if so2 and so2.startswith(PREBUILT) else '❌'} reload from cache: {dt2:.0f}s ({so2})")

## 3. Training-ready scene

If `datasets/scene1_ready.zip` is on Drive, it's unzipped and the rest of section 3 is skipped. Otherwise: install COLMAP, re-extract the frames, reuse the cached COLMAP model (or run COLMAP if there's none), undistort, and save the zip.

In [ ]:
import zipfile, imageio.v2 as imageio
SCENE_DIR = f'{WORK}/data/{SCENE}'
if not os.path.exists(f'{SCENE_DIR}/sparse/0/images.bin') and os.path.exists(READY_ZIP):
    with zipfile.ZipFile(READY_ZIP) as z:
        z.extractall(SCENE_DIR)
READY = os.path.exists(f'{SCENE_DIR}/sparse/0/images.bin')
print('scene restored from Drive' if READY else 'no ready scene on Drive yet: preparing it (first session only)')

In [ ]:
if READY:
    print('skipped: scene restored from Drive')
else:
    MAMBA   = f'{WORK}/bin/micromamba'
    SFM_ENV = f'{WORK}/sfm'
    os.environ['MAMBA_ROOT_PREFIX'] = f'{WORK}/mamba'
    if not os.path.exists(MAMBA):
        subprocess.run(f'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba',
                       shell=True, check=True)

    COLMAP = f'{SFM_ENV}/bin/colmap'
    SFM_ENVVARS = {**os.environ, 'QT_QPA_PLATFORM': 'offscreen'}

    def colmap_banner():
        r = subprocess.run([COLMAP, '-h'], capture_output=True, text=True, env=SFM_ENVVARS)
        return r.returncode, r.stdout + r.stderr

    # Known-good combo (see docs/failures.md F1.1). Don't add the separate `glomap` package: it breaks this build.
    if not os.path.exists(COLMAP) or colmap_banner()[0] != 0:
        shutil.rmtree(SFM_ENV, ignore_errors=True); t = time.time()
        subprocess.run([MAMBA, 'create', '-y', '-q', '-p', SFM_ENV, '-c', 'conda-forge',
                        'colmap=4.2.1=*cuda*', 'libfaiss=*=*cuda*'], check=True)
        print(f'installed in {time.time()-t:.0f}s')

    rc, hdr = colmap_banner()
    print('\n'.join(hdr.splitlines()[:2]))
    if rc != 0:
        raise RuntimeError('COLMAP does not start, see the error above')   # never mistake this for "CPU-only"
    COLMAP_CUDA = 'with CUDA' in hdr
    # COLMAP 4 may include GLOMAP's global SfM as a built-in command
    HAS_GLOBAL_MAPPER = subprocess.run([COLMAP, 'global_mapper', '-h'], capture_output=True, env=SFM_ENVVARS).returncode == 0
    print('COLMAP CUDA:', COLMAP_CUDA, '| global_mapper:', HAS_GLOBAL_MAPPER)
    if not COLMAP_CUDA:
        print('⚠️ COLMAP runs but without CUDA: SIFT + exhaustive matching will be slow (~30-60 min).')

In [ ]:
if READY:
    print('skipped: scene restored from Drive')
else:
    # helpers: run a command with logging, and pick option names that exist in this COLMAP version
    SFM_LOG = f'{OUT}/sfm_log.txt'

    def sh(cmd, tail=12, cwd=None):
        t = time.time()
        p = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, env=SFM_ENVVARS, cwd=cwd)
        out = p.stdout + p.stderr
        with open(SFM_LOG, 'a') as f:
            f.write(f'\n$ {" ".join(map(str, cmd))}\n{out}\n')
        dt = time.time() - t
        lines = [l for l in out.strip().splitlines() if l.strip()]
        if p.returncode != 0:
            print('\n'.join(lines[-40:]))
            raise RuntimeError(f'{os.path.basename(str(cmd[0]))} {cmd[1]} failed (exit {p.returncode}), full log: {SFM_LOG}')
        if tail: print('\n'.join(lines[-tail:]))
        print(f'✔ {os.path.basename(str(cmd[0]))} {cmd[1]}: {dt/60:.1f} min')
        return out, dt

    _help_cache = {}
    def opt(sub, *candidates):
        # COLMAP renamed some options between versions (e.g. SiftExtraction.use_gpu -> FeatureExtraction.use_gpu)
        if sub not in _help_cache:
            r = subprocess.run([COLMAP, sub, '-h'], capture_output=True, text=True, env=SFM_ENVVARS)
            _help_cache[sub] = r.stdout + r.stderr
        for c in candidates:
            if f'--{c}' in _help_cache[sub]:
                return c
        return None
    print('extract gpu opt:', opt('feature_extractor', 'FeatureExtraction.use_gpu', 'SiftExtraction.use_gpu'))
    print('match gpu opt:  ', opt('exhaustive_matcher', 'FeatureMatching.use_gpu', 'SiftMatching.use_gpu'))

In [ ]:
if READY:
    print('skipped: scene restored from Drive')
else:
    candidates = [f'{PROJECT}/videos/{SCENE}{ext}' for ext in ('.mp4', '.MP4', '.mov', '.MOV')]
    VIDEO = next((v for v in candidates if os.path.exists(v)), None)
    if VIDEO is None:
        raise FileNotFoundError(f'Upload your video to Drive as {candidates[0]} (or change SCENE in section 1)')
    LOCAL_VIDEO = f'{WORK}/{os.path.basename(VIDEO)}'
    if not os.path.exists(LOCAL_VIDEO):
        shutil.copy(VIDEO, LOCAL_VIDEO)

    probe = json.loads(subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-of', 'json', '-show_entries',
         'stream=width,height,codec_name,avg_frame_rate,color_transfer,pix_fmt:format=duration,size', LOCAL_VIDEO],
        capture_output=True, text=True, check=True).stdout)
    st = probe['streams'][0]
    duration = float(probe['format']['duration'])
    num, den = st['avg_frame_rate'].split('/')
    src_fps = float(num) / float(den)
    print(f"{st['width']}x{st['height']} | {st['codec_name']} {st.get('pix_fmt')} | {src_fps:.1f} fps | {duration:.0f} s | "
          f"{int(probe['format']['size'])/1e6:.0f} MB | color: {st.get('color_transfer', '?')}")

    if st.get('color_transfer') in ('smpte2084', 'arib-std-b67'):
        print('⚠️ HDR video: frames may look washed out. Re-film with HDR off if they do (section 4 shows samples).')
    if max(st['width'], st['height']) < 1280:
        print('⚠️ Low resolution: is this the original phone file? Fine detail will be limited. Use the original if you have it.')
    if duration < 20:
        print('⚠️ Very short video: probably too few viewpoints.')
    if duration > 300:
        print('⚠️ Long video: frames will be far apart. Keep it around 1-3 minutes for a first run.')

In [ ]:
if READY:
    print('skipped: scene restored from Drive')
else:
    import numpy as np, imageio.v2 as imageio, matplotlib.pyplot as plt

    def blur_score(path):
        g = imageio.imread(path).astype(np.float32)
        g = g[..., :3].mean(-1) if g.ndim == 3 else g
        lap = g[1:-1, 2:] + g[1:-1, :-2] + g[2:, 1:-1] + g[:-2, 1:-1] - 4 * g[1:-1, 1:-1]
        return float(lap.var())

    def pick_sharpest(scores, window, min_ratio=0.25):
        # the best frame per window, minus any that are far blurrier than the typical pick
        scores = np.asarray(scores, dtype=float)
        picks = [s + int(np.argmax(scores[s:s + window])) for s in range(0, len(scores), window)]
        floor = min_ratio * np.median(scores[picks])
        return [i for i in picks if scores[i] >= floor]

    RAW    = f'{WORK}/{SCENE}/raw_frames'
    IMAGES = f'{WORK}/{SCENE}/frames'
    if not (os.path.isdir(IMAGES) and os.listdir(IMAGES)):
        shutil.rmtree(RAW, ignore_errors=True); shutil.rmtree(IMAGES, ignore_errors=True)
        os.makedirs(RAW); os.makedirs(IMAGES)
        fps = min(src_fps, TARGET_FRAMES * OVERSAMPLE / duration)
        side = min(MAX_SIDE, max(st['width'], st['height']))
        t = time.time()
        subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', LOCAL_VIDEO, '-an',
                        '-vf', f'fps={fps:.4f},scale={side}:{side}:force_original_aspect_ratio=decrease',
                        '-qscale:v', '2', f'{RAW}/%05d.jpg'], check=True)
        raw = sorted(os.listdir(RAW))
        print(f'extracted {len(raw)} frames at {fps:.2f} fps in {time.time()-t:.0f}s')
        scores = [blur_score(f'{RAW}/{n}') for n in raw]
        keep = pick_sharpest(scores, OVERSAMPLE)
        for i in keep:
            shutil.copy(f'{RAW}/{raw[i]}', f'{IMAGES}/{raw[i]}')
        print(f'kept {len(keep)} of {len(raw)} frames')
        plt.figure(figsize=(14, 2.5)); plt.plot(scores, lw=0.7, label='all frames')
        plt.plot(keep, np.array(scores)[keep], '.', ms=3, label='kept'); plt.xlabel('frame'); plt.ylabel('sharpness')
        plt.legend(); plt.title('Sharpness over the video'); plt.savefig(f'{OUT}/sharpness.png', dpi=100, bbox_inches='tight'); plt.show()
        shutil.rmtree(RAW)

    files = sorted(os.listdir(IMAGES))
    fig, ax = plt.subplots(1, 5, figsize=(18, 3.5))
    for a, n in zip(ax, files[:: max(1, len(files) // 5)]):
        a.imshow(imageio.imread(f'{IMAGES}/{n}')); a.set_title(n, fontsize=8); a.axis('off')
    plt.show()
    print(len(files), 'frames | size', imageio.imread(f'{IMAGES}/{files[0]}').shape)

In [ ]:
if READY:
    print('skipped: scene restored from Drive')
else:
    import struct
    SFM = f'{WORK}/{SCENE}/sfm'
    DB  = f'{SFM}/database.db'
    os.makedirs(SFM, exist_ok=True)
    CACHE = f'{OUT}/sparse_colmap'
    timings = json.load(open(f'{OUT}/timings.json')) if os.path.exists(f'{OUT}/timings.json') else {}

    def n_images_in(model_dir):
        with open(f'{model_dir}/images.bin', 'rb') as f:
            return struct.unpack('<Q', f.read(8))[0]

    other_models = []
    if os.path.exists(f'{CACHE}/images.bin'):
        print('✔ using cached COLMAP model from Drive'); BEST = CACHE
    else:
        open(SFM_LOG, 'w').close()
        if os.path.exists(DB): os.remove(DB)
        shutil.rmtree(f'{SFM}/sparse_raw', ignore_errors=True)
        G = '1' if COLMAP_CUDA else '0'

        cmd = [COLMAP, 'feature_extractor', '--database_path', DB, '--image_path', IMAGES,
               '--ImageReader.single_camera', '1', '--ImageReader.camera_model', 'OPENCV']
        gpu = opt('feature_extractor', 'FeatureExtraction.use_gpu', 'SiftExtraction.use_gpu')
        if gpu: cmd += [f'--{gpu}', G]
        _, timings['extract_s'] = sh(cmd)

        def match(kind, loop=True):
            sub = f'{kind}_matcher'
            cmd = [COLMAP, sub, '--database_path', DB]
            gpu = opt(sub, 'FeatureMatching.use_gpu', 'SiftMatching.use_gpu')
            if gpu: cmd += [f'--{gpu}', G]
            if kind == 'sequential':
                ov = opt(sub, 'SequentialMatching.overlap')
                if ov: cmd += [f'--{ov}', '20']
                ld = opt(sub, 'SequentialMatching.loop_detection')
                if ld and loop: cmd += [f'--{ld}', '1']
            return sh(cmd)[1]

        if MATCHER == 'sequential':
            try:
                timings['match_s'] = match('sequential')
            except RuntimeError as e:
                print('Loop detection failed, matching neighbours only:', e)
                timings['match_s'] = match('sequential', loop=False)
        else:
            timings['match_s'] = match('exhaustive')

        os.makedirs(f'{SFM}/sparse_raw')
        cmd = [COLMAP, 'mapper', '--database_path', DB, '--image_path', IMAGES, '--output_path', f'{SFM}/sparse_raw']
        tol = opt('mapper', 'Mapper.ba_global_function_tolerance')
        if tol: cmd += [f'--{tol}', '0.000001']
        _, timings['mapper_s'] = sh(cmd, tail=5)

        models = [os.path.dirname(m) for m in glob.glob(f'{SFM}/sparse_raw/*/images.bin')]
        if not models:
            raise RuntimeError('COLMAP built no model: the video probably has too little overlap or texture. See the capture guide.')
        print('models found (images in each):', sorted((n_images_in(m) for m in models), reverse=True))
        BEST = max(models, key=n_images_in)
        other_models = [n_images_in(m) for m in models if m != BEST]
        shutil.copytree(BEST, CACHE, dirs_exist_ok=True)
        json.dump(timings, open(f'{OUT}/timings.json', 'w'), indent=1)
    print('timings (min):', {k: round(v / 60, 1) for k, v in timings.items()})
    print('registered', n_images_in(BEST), 'of', len(os.listdir(IMAGES)), 'frames')
    # the cached model must belong to these frames (same video, same frame settings)
    import struct
    posed = []
    with open(f'{BEST}/images.bin', 'rb') as f:
        for _ in range(struct.unpack('<Q', f.read(8))[0]):
            f.read(4 + 32 + 24 + 4)                                   # id, rotation, translation, camera id
            name = b''
            while (c := f.read(1)) != b'\x00':
                name += c
            posed.append(name.decode())
            f.seek(24 * struct.unpack('<Q', f.read(8))[0], 1)         # skip 2D points
    missing = sorted(set(posed) - set(os.listdir(IMAGES)))
    print(f'COLMAP model: {len(posed)} poses | frames on disk: {len(os.listdir(IMAGES))} | posed frames missing: {len(missing)}')
    assert not missing, f'The cached COLMAP model does not match these frames (e.g. {missing[:3]}): the video or frame settings changed. Delete {OUT}/sparse_colmap and run again.'

In [ ]:
if READY:
    print('skipped: scene restored from Drive')
else:
    SCENE_DIR = f'{WORK}/data/{SCENE}'
    if not os.path.exists(f'{SCENE_DIR}/sparse/0/images.bin'):
        shutil.rmtree(SCENE_DIR, ignore_errors=True)
        os.makedirs(SCENE_DIR, exist_ok=True)      # COLMAP aborts if the parent folder is missing
        sh([COLMAP, 'image_undistorter', '--image_path', IMAGES, '--input_path', BEST,
            '--output_path', SCENE_DIR, '--output_type', 'COLMAP'], tail=3)
        os.makedirs(f'{SCENE_DIR}/sparse/0', exist_ok=True)
        for fn in glob.glob(f'{SCENE_DIR}/sparse/*.bin'):
            shutil.move(fn, f'{SCENE_DIR}/sparse/0/')
    print(len(os.listdir(f'{SCENE_DIR}/images')), 'images | size', imageio.imread(glob.glob(f'{SCENE_DIR}/images/*')[0]).shape)
    with zipfile.ZipFile(READY_ZIP + '.part', 'w', zipfile.ZIP_STORED) as z:     # JPEGs don't compress further
        for root, _, files in os.walk(SCENE_DIR):
            for fn in files:
                z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), SCENE_DIR))
    os.replace(READY_ZIP + '.part', READY_ZIP)
    print(f'saved {READY_ZIP} ({os.path.getsize(READY_ZIP)/1e6:.0f} MB)')

In [ ]:
DATA = SCENE_DIR
imgs = sorted(glob.glob(f'{DATA}/images/*'))
print(len(imgs), 'training-ready images | size', imageio.imread(imgs[0]).shape)

## 4. Train (30k steps, scored at 7k and 30k)

Every 8th frame is held out, as in Phase 1b.

In [ ]:
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'default', '--help'], capture_output=True,
                          text=True, cwd=f'{REPO}/examples').stdout
assert re.search(r'lpips.net', help_txt), 'this gsplat has no --lpips_net option'

def step_of(path):
    return int(re.search(r'step(\d+)', os.path.basename(path)).group(1))

def stats_files(rd, stage):
    return sorted(glob.glob(f'{rd}/stats/{stage}_step*.json'), key=step_of)    # numeric order: 6999 before 29999

def check_resolution(rd, data_dir):
    expected = imageio.imread(sorted(glob.glob(f'{data_dir}/images/*'))[0]).shape[1]
    actual = imageio.imread(sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[0]).shape[1] / 2   # GT | render
    if abs(actual - expected) / expected > 0.1:
        raise RuntimeError(f'Trained at {actual:.0f}px instead of {expected}px: wrong data_factor? See F1.2')
    return int(actual)

def train(exp):
    rd = f'{PROJECT}/results/{run_name(exp)}'
    done = [f for f in stats_files(rd, 'val') if step_of(f) == STEPS - 1]
    if done:
        print(f'✔ {run_name(exp)}: already trained'); return
    if os.path.exists(rd):                      # a half-finished run can't be resumed: keep its files, start fresh
        shutil.move(rd, rd + time.strftime('_failed_%Y%m%d_%H%M'))
    cmd = [sys.executable, 'simple_trainer.py', *EXPERIMENTS[exp], '--data_dir', DATA, '--data_factor', '1',
           '--result_dir', rd, '--max_steps', str(STEPS), '--disable_viewer', '--lpips_net', 'vgg']
    os.makedirs(rd, exist_ok=True)
    print('$', ' '.join(cmd[1:]))
    t0 = last = time.time(); buf = ''; tail = []
    env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}   # less memory fragmentation (F2.1)
    p = subprocess.Popen(cmd, cwd=f'{REPO}/examples', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f'{rd}/train_log.txt', 'w') as lf:
        while (chunk := p.stdout.read(512)):
            lf.write(chunk); buf += chunk
            parts = re.split(r'[\r\n]', buf); buf = parts[-1]
            for ln in filter(str.strip, parts[:-1]):
                tail = (tail + [ln])[-40:]
                if 'PSNR' in ln or time.time() - last > 120:
                    print(f'[{(time.time()-t0)/60:5.1f} min] {ln.strip()[:150]}'); last = time.time()
    if p.wait() != 0:
        print('\n'.join(tail)); raise RuntimeError(f'training failed, log: {rd}/train_log.txt')
    json.dump({'wall_min': round((time.time() - t0) / 60, 1)}, open(f'{rd}/wall_time.json', 'w'))
    print(f'✔ {run_name(exp)} finished in {(time.time()-t0)/60:.1f} min | trained at width {check_resolution(rd, DATA)}px')

for exp in RUN_NOW:
    train(exp)

## 5. Results vs Phase 1b

Same frames, same test split. Phase 1b used the default strategy for 7k steps: PSNR 29.76, SSIM 0.951, LPIPS 0.166, 541,980 Gaussians.

In [ ]:
import numpy as np, csv, datetime
PUBLISHED = {'psnr': 29.76, 'ssim': 0.951, 'lpips': 0.166}     # Phase 1b: default strategy, 7k steps
BYTES_PER_GS = 62 * 4

rows = []
for exp in EXPERIMENTS:
    rd = f'{PROJECT}/results/{run_name(exp)}'
    vals = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'val')}
    trains = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'train')}
    for step in sorted(vals):
        v, tr = vals[step], trains.get(step, {})
        rows.append({'exp': exp, 'steps': step + 1, 'psnr': v['psnr'], 'ssim': v['ssim'], 'lpips': v['lpips'],
                     'num_GS': v['num_GS'], 'train_min': tr.get('ellipse_time', np.nan) / 60, 'mem_gb': tr.get('mem', np.nan),
                     'fps': 1 / v['ellipse_time'], 'ply_mb': v['num_GS'] * BYTES_PER_GS / 1e6})
if not rows:
    print('No finished runs yet.')
else:
    print(f"{'experiment':>10} | {'steps':>6} | {'PSNR':>6} | {'SSIM':>6} | {'LPIPS':>6} | {'#Gaussians':>10} | "
          f"{'train':>7} | {'VRAM':>6} | {'FPS':>5} | {'.ply':>7} | vs Phase 1b")
    for r in rows:
        gap = r['psnr'] - PUBLISHED['psnr']
        verdict = f'{gap:+.2f} dB'
        print(f"{r['exp']:>10} | {r['steps']:>6} | {r['psnr']:6.2f} | {r['ssim']:6.3f} | {r['lpips']:6.3f} | {r['num_GS']:>10,} | "
              f"{r['train_min']:5.1f} m | {r['mem_gb']:4.1f}GB | {r['fps']:5.1f} | {r['ply_mb']:5.0f}MB | {verdict}")
    print(f"{'phase 1b':>10} | {7000:>6} | {PUBLISHED['psnr']:6.2f} | {PUBLISHED['ssim']:6.3f} | {PUBLISHED['lpips']:6.3f} |")
    json.dump(rows, open(f'{FIG}/phase2_results.json', 'w'), indent=1)

    # shared metrics.csv: one row per finished 30k run
    CSV = f'{PROJECT}/results/metrics.csv'
    header = next(csv.reader(open(CSV))) if os.path.exists(CSV) else \
        ['date', 'run', 'scene', 'res', 'strategy', 'steps', 'num_gaussians', 'psnr', 'ssim', 'lpips', 'gpu', 'mem_gb', 'ply_mb']
    final = [r for r in rows if r['steps'] == STEPS]
    names = {run_name(r['exp']) for r in final}
    old = [r for r in csv.DictReader(open(CSV)) if r['run'] not in names] if os.path.exists(CSV) else []
    with open(CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=header, extrasaction='ignore'); w.writeheader(); w.writerows(old)
        for r in final:
            w.writerow({'date': str(datetime.date.today()), 'run': run_name(r['exp']), 'scene': SCENE, 'res': 'x'.join(map(str, imageio.imread(imgs[0]).shape[1::-1])),
                        'strategy': ' '.join(EXPERIMENTS[r['exp']]), 'steps': STEPS, 'num_gaussians': r['num_GS'],
                        'psnr': round(r['psnr'], 3), 'ssim': round(r['ssim'], 4), 'lpips': f"{round(r['lpips'], 4)} (vgg)",
                        'gpu': torch.cuda.get_device_name(0), 'mem_gb': round(r['mem_gb'], 2), 'ply_mb': round(r['ply_mb'], 1)})
    print('\nlogged', len(final), 'finished run(s) to', CSV)

## 6. Render grids: real | render | error

In [ ]:
import matplotlib.pyplot as plt
finished = [e for e in EXPERIMENTS if any(r['exp'] == e and r['steps'] == STEPS for r in rows)]
VIEWS = [0, 8, 16]
for exp in finished:
    rd = f'{PROJECT}/results/{run_name(exp)}'
    fig, ax = plt.subplots(len(VIEWS), 3, figsize=(18, 4 * len(VIEWS)))
    for i, v in enumerate(VIEWS):
        im = imageio.imread(f'{rd}/renders/val_step{STEPS - 1}_{v:04d}.png')[..., :3].astype(np.float32) / 255
        w = im.shape[1] // 2; gt, pr = im[:, :w], im[:, w:]
        err = np.abs(gt - pr).mean(-1)
        psnr = -10 * np.log10(((gt - pr) ** 2).mean())
        ax[i, 0].imshow(gt); ax[i, 0].set_title(f'real (test view {v})')
        ax[i, 1].imshow(pr); ax[i, 1].set_title(f'{exp} render, PSNR {psnr:.2f}')
        ax[i, 2].imshow(err, cmap='inferno', vmin=0, vmax=0.25); ax[i, 2].set_title('error (black = exact)')
    for a in ax.flat: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{FIG}/grid_{exp}.jpg', dpi=80); plt.show()

# same crop from every finished run, to compare fine detail
if len(finished) > 1:
    fig, ax = plt.subplots(1, len(finished) + 1, figsize=(5 * (len(finished) + 1), 5))
    for j, exp in enumerate(finished):
        im = imageio.imread(f'{PROJECT}/results/{run_name(exp)}/renders/val_step{STEPS - 1}_0000.png')[..., :3]
        w = im.shape[1] // 2; h = im.shape[0]
        crop = lambda x: x[h // 3: h // 3 + 300, w // 3: w // 3 + 300]
        if j == 0: ax[0].imshow(crop(im[:, :w])); ax[0].set_title('real')
        ax[j + 1].imshow(crop(im[:, w:])); ax[j + 1].set_title(exp)
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{FIG}/crop_compare.jpg', dpi=90); plt.show()
print('figures in', FIG)

## 7. Export for SuperSplat

This is your own scene, so there's no licence limit on sharing it. Set `DOWNLOAD = False` to keep it on Drive only.

In [ ]:
from plyfile import PlyData, PlyElement
EXPORT = 'mcmc_1M'          # ⬅️ which finished run to export
DOWNLOAD = True             # False = keep it on Drive only;  True = also download the file to this computer (uses your internet)

def ckpt_to_ply(ckpt_path, ply_path):
    s = torch.load(ckpt_path, map_location='cpu', weights_only=False)['splats']
    means = s['means'].numpy()
    sh0 = s['sh0'].transpose(1, 2).flatten(1).numpy()            # [N,1,3] -> [N,3]
    shN = s['shN'].transpose(1, 2).flatten(1).numpy()            # [N,K,3] -> [N,3K] (channel-major, 3DGS convention)
    opac = s['opacities'].numpy()[:, None]                       # logit space
    scales = s['scales'].numpy()                                 # log space
    quats = s['quats'].numpy()                                   # wxyz
    normals = np.zeros_like(means)
    attrs = ['x','y','z','nx','ny','nz'] + [f'f_dc_{i}' for i in range(sh0.shape[1])] \
          + [f'f_rest_{i}' for i in range(shN.shape[1])] + ['opacity'] \
          + [f'scale_{i}' for i in range(3)] + [f'rot_{i}' for i in range(4)]
    data = np.concatenate([means, normals, sh0, shN, opac, scales, quats], axis=1).astype(np.float32)
    el = np.empty(data.shape[0], dtype=[(a, 'f4') for a in attrs])
    for i, a in enumerate(attrs):
        el[a] = data[:, i]
    PlyData([PlyElement.describe(el, 'vertex')]).write(ply_path)
    return data.shape[0]

rd = f'{PROJECT}/results/{run_name(EXPORT)}'
ckpts = sorted(glob.glob(f'{rd}/ckpts/ckpt_*_rank0.pt'), key=lambda p: int(re.search(r'ckpt_(\d+)', p).group(1)))
assert ckpts, f'no checkpoint for {EXPORT}: train it first'
PLY = f'{PROJECT}/exports/{run_name(EXPORT)}.ply'
n = ckpt_to_ply(ckpts[-1], PLY)
print(f'{n:,} Gaussians from {os.path.basename(ckpts[-1])} -> {PLY} ({os.path.getsize(PLY)/1e6:.0f} MB)')
if DOWNLOAD:
    from google.colab import files
    files.download(PLY)

## ✅ Phase 2b checklist
- [ ] Section 2 printed `✅ gsplat CUDA loaded from the Drive cache`
- [ ] `datasets/scene1_ready.zip` saved (first session)
- [ ] 30k scores vs Phase 1b in section 5
- [ ] In SuperSplat: is the table sharper than the Phase 1b `.ply`? Are there fewer floaters?
- [ ] Anything that broke → `docs/failures.md`

**Paste back into the chat:** section 5's table, and SuperSplat screenshots from roughly the same views as Phase 1b's.

**Next (Phase 3):** clean this model up in SuperSplat, then build the web viewer.